## Data Preparation
 
Both datasets use different genre tag formats. All genres must be normalised into the same 12-label vocabulary, texts cleaned, and data split proportionally.

### 1.1 Download Datasets

### 1.2 Parse CMU Book Genres
Extracts human-readable names into a plain Python list that the mapping step can work with.

In [ ]:
import pandas as pd
import ast

cmu = pd.read_csv("booksummaries.txt", sep="\t", header=None,
    names=["wiki_id","freebase_id","title","author","pub_date","genres","summary"])

# Diagnostic: inspect raw genre format before parsing
print(repr(cmu["genres"].dropna().iloc[0]))  # See exact format

def parse_cmu_genres(genre_str):
    # Uses ast.literal_eval, NOT json.loads, because:
    # 1. CMU genres are Python dict literals with single quotes
    # 2. Genre names can contain apostrophes (e.g. Children's literature)
    #    which break the single-quote replace hack used with json.loads
    # 3. Older CMU entries have Python 2 u'..' prefix, rejected by json
    if pd.isna(genre_str) or genre_str.strip() == "{}":
        return []
    try:
        return list(ast.literal_eval(genre_str).values())
    except:
        return []

cmu["genre_list"] = cmu["genres"].apply(parse_cmu_genres)
print(f"CMU rows: {len(cmu)}")


'{"/m/016lj8": "Roman \\u00e0 clef", "/m/06nbt": "Satire", "/m/0dwly": "Children\'s literature", "/m/014dfn": "Speculative fiction", "/m/02xlf": "Fiction"}'
CMU rows: 16559


### 1.3 Parse TMDB Movie Genres
Extract plain genre name lists so both datasets are in the same format before mapping.

In [ ]:
import ast

movies = pd.read_csv("movies_metadata.csv", low_memory=False)
movies = movies[["title", "overview", "genres"]].dropna()

def parse_tmdb_genres(genre_str):
    if pd.isna(genre_str): return []
    try: return [g["name"] for g in ast.literal_eval(genre_str)]
    except: return []

movies["genre_list"] = movies["genres"].apply(parse_tmdb_genres)
movies = movies[movies["overview"].str.len() > 30]
print(f"Movies rows: {len(movies)}")


Movies rows: 44253


### 1.4 Genre Taxonomy Mapping
Maps the raw genre strings to 12 shared target genres. The lowercase strip is essential due to inconsistent casing.

In [ ]:
# Complete genre map -keys are ALL lowercase
GENRE_MAP = {
    "action": "Action", "adventure": "Action", "action/adventure": "Action",
    "spy film": "Action", "superhero": "Action", "martial arts film": "Action",
    "comedy": "Comedy", "romantic comedy": "Comedy", "satire": "Comedy",
    "black comedy": "Comedy", "parody": "Comedy", "humour": "Comedy",
    "drama": "Drama", "melodrama": "Drama", "historical fiction": "Drama",
    "war film": "Drama", "biographical film": "Drama", "legal drama": "Drama",
    "historical novel": "Drama", "historical drama": "Drama",
    "horror": "Horror", "horror fiction": "Horror", "gothic fiction": "Horror",
    "psychological horror": "Horror", "supernatural fiction": "Horror",
    "thriller": "Thriller", "crime fiction": "Thriller", "mystery": "Thriller",
    "detective fiction": "Thriller", "crime": "Thriller", "suspense": "Thriller",
    "noir fiction": "Thriller", "psychological thriller": "Thriller",
    "science fiction": "Sci-Fi", "cyberpunk": "Sci-Fi", "space opera": "Sci-Fi",
    "dystopia": "Sci-Fi", "dystopian fiction": "Sci-Fi", "hard science fiction": "Sci-Fi",
    "alternate history": "Sci-Fi",
    "fantasy": "Fantasy", "high fantasy": "Fantasy", "epic fantasy": "Fantasy",
    "urban fantasy": "Fantasy", "dark fantasy": "Fantasy", "fairy tale": "Fantasy",
    "romance novel": "Romance", "romance": "Romance", "love story": "Romance",
    "chick lit": "Romance", "romantic fiction": "Romance",
    "animation": "Animation", "animated film": "Animation", "family film": "Animation",
    "documentary": "Documentary", "docudrama": "Documentary",
    "literary fiction": "Literary Fiction", "fiction": "Literary Fiction",
    "bildungsroman": "Literary Fiction", "philosophical fiction": "Literary Fiction",
    "non-fiction": "Non-Fiction", "biography": "Non-Fiction",
    "autobiography": "Non-Fiction", "memoir": "Non-Fiction",
}

TARGET_GENRES = ["Action","Comedy","Drama","Horror","Thriller",
                 "Sci-Fi","Fantasy","Romance","Animation",
                 "Documentary","Literary Fiction","Non-Fiction"]

def map_genres(raw_list):
    mapped = set()
    for g in raw_list:
        key = g.strip().lower()  # lowercase before lookup — fixes casing issues
        if key in GENRE_MAP:
            mapped.add(GENRE_MAP[key])
    return list(mapped)

cmu["mapped_genres"]    = cmu["genre_list"].apply(map_genres)
movies["mapped_genres"] = movies["genre_list"].apply(map_genres)

cmu    = cmu[cmu["mapped_genres"].apply(len) > 0].reset_index(drop=True)
movies = movies[movies["mapped_genres"].apply(len) > 0].reset_index(drop=True)

print(f"CMU after mapping:    {len(cmu)}")
print(f"Movies after mapping: {len(movies)}")


CMU after mapping:    10163
Movies after mapping: 41473


### 1.5 Save Cleaned DataFrames
Saves original datasets paired by title for mismatch detection later.

In [ ]:
# Save cleaned dataframes
cmu[["title","summary","mapped_genres"]].to_csv("cmu_cleaned.csv", index=False)
movies[["title","overview","mapped_genres"]].to_csv("movies_cleaned.csv", index=False)
print("Saved: cmu_cleaned.csv")
print("Saved: movies_cleaned.csv")

# Files produced so far:
# cmu_cleaned.csv
# movies_cleaned.csv


Saved: cmu_cleaned.csv
Saved: movies_cleaned.csv


### 1.6 Multi-Label Binarization and Train/Val/Test Split
Uses `MultiLabelBinarizer` and `MultilabelStratifiedShuffleSplit`.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import numpy as np, pickle

mlb = MultiLabelBinarizer(classes=TARGET_GENRES)
mlb.fit([TARGET_GENRES])

cmu_df = pd.DataFrame({
    "text":       cmu["summary"].values,
    "source":     "book",
    "title":      cmu["title"].values,
    "labels":     list(mlb.transform(cmu["mapped_genres"]))
})
movie_df = pd.DataFrame({
    "text":   movies["overview"].values,
    "source": "movie",
    "title":  movies["title"].values,
    "labels": list(mlb.transform(movies["mapped_genres"]))
})
full_df = pd.concat([cmu_df, movie_df], ignore_index=True)
full_df["text"] = full_df["text"].str.strip().str.replace(r"\s+", " ", regex=True)
full_df = full_df[full_df["text"].str.len() > 30]

X = full_df["text"].values
y = np.stack(full_df["labels"].values)

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
for tv_idx, test_idx in msss.split(X, y):
    X_tv, X_test = X[tv_idx], X[test_idx]
    y_tv, y_test = y[tv_idx], y[test_idx]

msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.176, random_state=42)
for train_idx, val_idx in msss2.split(X_tv, y_tv):
    X_train, X_val = X_tv[train_idx], X_tv[val_idx]
    y_train, y_val = y_tv[train_idx], y_tv[val_idx]

with open("data_splits.pkl", "wb") as f:
    pickle.dump({"X_train":X_train,"y_train":y_train,
                 "X_val":X_val,"y_val":y_val,
                 "X_test":X_test,"y_test":y_test}, f)
with open("mlb.pkl", "wb") as f:
    pickle.dump(mlb, f)
print("Saved: data_splits.pkl")
print("Saved: mlb.pkl")


Saved: data_splits.pkl
Saved: mlb.pkl


## PHASE 2 Generate Embeddings (Topic Modeling Prep)
Uses SBERT `all-MiniLM-L6-v2` to avoid re-encoding on Kaggle. For BERTopic and Mismatch Detection.

In [ ]:
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

# Load splits
with open("data_splits.pkl", "rb") as f:
    splits = pickle.load(f)
X_train = splits["X_train"]
X_val   = splits["X_val"]
X_test  = splits["X_test"]

# Combine ALL texts in this fixed order
# all_embeddings[0:len(X_train)] = train embeddings
# all_embeddings[len(X_train):len(X_train)+len(X_val)] = val embeddings
# all_embeddings[len(X_train)+len(X_val):] = test embeddings
all_texts = list(X_train) + list(X_val) + list(X_test)
print(f"Total texts to embed: {len(all_texts)}")

# Load SBERT model (~80 MB download on first run)
sbert = SentenceTransformer("all-MiniLM-L6-v2")

# Encode- batch_size=64 works on CPU
embeddings = sbert.encode(
    all_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")
assert len(all_texts) == len(embeddings), "Count mismatch!"
np.save("all_embeddings.npy", embeddings)
print("Saved: all_embeddings.npy")


c:\Users\ab\Desktop\NLP_Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total texts to embed: 51618


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5244.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 807/807 [01:50<00:00,  7.29it/s]


Embeddings shape: (51618, 384)
Saved: all_embeddings.npy


## PHASE 3 Build Book-Movie Pairs
Matches CMU book titles to TMDB movie titles using fuzzy string matching.

In [ ]:
import pandas as pd
from rapidfuzz import fuzz, process

# Load the cleaned dataframes saved in Phase 1.5
cmu_clean    = pd.read_csv("cmu_cleaned.csv")
movies_clean = pd.read_csv("movies_cleaned.csv")

movie_titles = movies_clean["title"].str.lower().str.strip().tolist()

pairs = []
for i, row in cmu_clean.iterrows():
    book_title = str(row["title"]).lower().strip()
    result = process.extractOne(
        book_title, movie_titles, scorer=fuzz.token_sort_ratio
    )
    if result is None: continue
    match, score, idx = result
    if score > 85:   # Only high-confidence matches
        pairs.append({
            "book_title":   row["title"],
            "movie_title":  movies_clean.iloc[idx]["title"],
            "book_summary": row["summary"],
            "movie_plot":   movies_clean.iloc[idx]["overview"],
            "book_genres":  row["mapped_genres"],
            "movie_genres": movies_clean.iloc[idx]["mapped_genres"],
            "match_score":  score
        })

pairs_df = pd.DataFrame(pairs)
print(f"Matched book-movie pairs: {len(pairs_df)}")
pairs_df.to_csv("book_movie_pairs.csv", index=False)
print("Saved: book_movie_pairs.csv")


Matched book-movie pairs: 2392
Saved: book_movie_pairs.csv
